# NJ Housing Price Data Preparation

Generates training-ready JSONL splits for QLoRA fine-tuning of Qwen2.5-0.5B.

## What this notebook does
1. Imports shared `format_prompt()` from `lambda/prompt_utils.py`
2. Generates synthetic NJ housing records using county-level log-normal price distributions
3. Loads real NJ housing records from SR1A 2024 public data (>=30% of total)
4. Validates price distribution against known NJ medians
5. Exports stratified 70/15/15 JSONL splits to `data/splits/`

## Requirements
- Run from `notebooks/` directory OR set `repo_root` to repo root
- SR1A 2024 file placed at `data/raw/SR1A_2024.txt` (see data/SCHEMA.md for source)
- `pip install -r requirements.txt` completed

## Import Note
The `lambda/` directory name matches the AWS Lambda deployment target but clashes with
Python's `lambda` reserved keyword. We use `importlib.import_module('lambda.prompt_utils')`
instead of `from lambda.prompt_utils import ...` (which raises SyntaxError).

In [ ]:
import sys
import os
import json
import importlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for saved plots
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split

# Add repo root to path so lambda/ package is importable
# Works whether notebook is run from repo root or from notebooks/
_nb_dir = os.getcwd()
repo_root = _nb_dir if os.path.isdir(os.path.join(_nb_dir, "lambda")) else os.path.dirname(_nb_dir)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# NOTE: 'lambda' is a Python reserved keyword — cannot use 'from lambda.prompt_utils import ...'
# Must use importlib.import_module() to load from the lambda/ package.
_pu = importlib.import_module('lambda.prompt_utils')
format_prompt = _pu.format_prompt
parse_price_from_output = _pu.parse_price_from_output

# Verify import works and prompt format is correct
_test_prompt = format_prompt(3, 2.0, 1500, 0.25, 1990, "07650", "Single Family")
assert _test_prompt.endswith("Predicted price: $"), f"Unexpected prompt format: {_test_prompt!r}"
print(f"format_prompt imported and verified.")
print(f"Sample prompt: {_test_prompt}")

# Paths
DATA_RAW = os.path.join(repo_root, "data", "raw")
DATA_SPLITS = os.path.join(repo_root, "data", "splits")
os.makedirs(DATA_SPLITS, exist_ok=True)

# Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

In [ ]:
# NJ county log-normal price parameters (2024-2025)
# Source: ATTOM Sept 2025 + NJRealtors 2024 year-end report
# mu = log(county_median), sigma calibrated to observed NJ price spread
# zip_prefix: list of 3-digit zip prefixes common in this county
COUNTY_PRICE_PARAMS = {
    "Bergen":      {"mu": np.log(721200), "sigma": 0.45, "zip_prefix": ["070", "071", "074", "076", "077"]},
    "Monmouth":    {"mu": np.log(702500), "sigma": 0.45, "zip_prefix": ["077", "087"]},
    "Morris":      {"mu": np.log(665000), "sigma": 0.40, "zip_prefix": ["079"]},
    "Essex":       {"mu": np.log(660900), "sigma": 0.50, "zip_prefix": ["070", "071", "072"]},
    "Hudson":      {"mu": np.log(659200), "sigma": 0.45, "zip_prefix": ["070", "073"]},
    "Somerset":    {"mu": np.log(624300), "sigma": 0.38, "zip_prefix": ["088", "089"]},
    "Union":       {"mu": np.log(588700), "sigma": 0.42, "zip_prefix": ["070", "072", "079"]},
    "Hunterdon":   {"mu": np.log(574800), "sigma": 0.35, "zip_prefix": ["088"]},
    "Passaic":     {"mu": np.log(569100), "sigma": 0.43, "zip_prefix": ["074", "075", "077"]},
    "Cape May":    {"mu": np.log(545400), "sigma": 0.50, "zip_prefix": ["082"]},
    "Middlesex":   {"mu": np.log(540800), "sigma": 0.40, "zip_prefix": ["088", "089"]},
    "Ocean":       {"mu": np.log(506000), "sigma": 0.42, "zip_prefix": ["087", "082"]},
    "Sussex":      {"mu": np.log(429700), "sigma": 0.38, "zip_prefix": ["074", "078"]},
    "Mercer":      {"mu": np.log(410100), "sigma": 0.42, "zip_prefix": ["085", "086"]},
    "Burlington":  {"mu": np.log(395500), "sigma": 0.38, "zip_prefix": ["080", "081", "083", "084", "085", "086"]},
    "Atlantic":    {"mu": np.log(360200), "sigma": 0.48, "zip_prefix": ["082", "083"]},
    "Gloucester":  {"mu": np.log(361100), "sigma": 0.38, "zip_prefix": ["080", "081", "083"]},
    "Camden":      {"mu": np.log(341700), "sigma": 0.42, "zip_prefix": ["080", "081"]},
    "Warren":      {"mu": np.log(340000), "sigma": 0.40, "zip_prefix": ["078", "088"]},
    "Salem":       {"mu": np.log(259700), "sigma": 0.40, "zip_prefix": ["080", "081"]},
    "Cumberland":  {"mu": np.log(251200), "sigma": 0.42, "zip_prefix": ["083"]},
}

PROPERTY_TYPES = ["Single Family", "Condo", "Townhouse", "Multi-Family"]
PROPERTY_TYPE_PROBS = [0.60, 0.20, 0.15, 0.05]

print(f"Loaded price parameters for {len(COUNTY_PRICE_PARAMS)} NJ counties.")

In [ ]:
def generate_synthetic_record(county: str, rng: np.random.Generator) -> dict:
    """
    Generate one synthetic NJ housing record using county-level log-normal price distribution.
    Price is generated first; all other features are derived from price.
    This produces realistic feature-price correlations (vs. generating features independently).
    """
    params = COUNTY_PRICE_PARAMS[county]
    price = float(rng.lognormal(params["mu"], params["sigma"]))
    price = max(100_000.0, min(price, 3_000_000.0))  # clip outliers

    # Derive sqft from price (NJ price/sqft range: $250-700)
    price_per_sqft = float(rng.uniform(250, 700))
    sqft = int(price / price_per_sqft)
    sqft = max(500, min(sqft, 8000))

    # Derive bedrooms from sqft (roughly 1 bed per 500 sqft, +/-1)
    bedrooms = min(6, max(1, int(sqft / 500) + int(rng.integers(-1, 2))))

    # Bathrooms correlated with bedrooms (0.5 increments)
    bathrooms = round(float(bedrooms * rng.uniform(0.5, 1.2) * 0.5) * 2) / 2
    bathrooms = max(1.0, min(bathrooms, 5.0))

    # Lot size: log-normal centered on 0.25 acres
    lot_size = round(float(rng.lognormal(np.log(0.25), 0.8)), 2)
    lot_size = max(0.05, min(lot_size, 10.0))

    year_built = int(rng.integers(1900, 2025))

    # Sample zip from county prefix (5-digit string with leading zero preserved)
    # NJ zips are 5 digits: e.g., 07650. Prefix is 3 chars + 2 digit suffix = 5 total
    prefix = str(rng.choice(params["zip_prefix"]))
    suffix = str(int(rng.integers(10, 100))).zfill(2)
    zip_code = prefix + suffix  # 3 + 2 = 5 digits

    property_type = str(rng.choice(PROPERTY_TYPES, p=PROPERTY_TYPE_PROBS))

    return {
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "sqft": sqft,
        "lot_size": lot_size,
        "year_built": year_built,
        "zip_code": zip_code,
        "property_type": property_type,
        "price": round(price / 100) * 100,  # round to nearest $100
        "source": "synthetic",
        "county": county,
    }


def validate_price_distribution(df: pd.DataFrame, save_path: str = None) -> None:
    """
    Validate generated price distribution against known NJ statewide median.
    Raises ValueError if generated median differs from $560,000 by more than 20%.
    """
    prices = df["price"].values
    known_nj_median = 560_000
    generated_median = float(np.median(prices))
    pct_diff = abs(generated_median - known_nj_median) / known_nj_median * 100

    print(f"Generated median:  ${generated_median:,.0f}")
    print(f"NJ 2024 median:    ${known_nj_median:,.0f}")
    print(f"Difference:        {pct_diff:.1f}%")

    if pct_diff > 20:
        raise ValueError(
            f"Generated median ${generated_median:,.0f} differs from NJ median "
            f"${known_nj_median:,.0f} by {pct_diff:.1f}% > 20% threshold. "
            "Adjust county distribution parameters (mu/sigma) in COUNTY_PRICE_PARAMS."
        )

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(np.log(prices), bins=50, alpha=0.7, color="steelblue", label="Generated")
    axes[0].axvline(np.log(known_nj_median), color="red", linestyle="--", label=f"NJ median (log={np.log(known_nj_median):.2f})")
    axes[0].set_xlabel("log(price)")
    axes[0].set_title("Log-Price Distribution")
    axes[0].legend()

    axes[1].hist(prices / 1000, bins=50, alpha=0.7, color="steelblue")
    axes[1].axvline(known_nj_median / 1000, color="red", linestyle="--", label=f"NJ median ${known_nj_median/1000:.0f}k")
    axes[1].set_xlabel("Price ($1000s)")
    axes[1].set_title("Price Distribution")
    axes[1].legend()

    plt.tight_layout()
    out = save_path or os.path.join(repo_root, "data", "price_distribution_check.png")
    plt.savefig(out, dpi=100)
    plt.close()
    print(f"Distribution plot saved to {out}")
    print("PASSED: Price distribution within 20% tolerance.")


print("Synthetic generation functions defined.")

In [ ]:
# Generate synthetic records
# Target: 5,000-10,000 total records; 70%+ synthetic to allow 30%+ real
# Generate 7,000 synthetic; real data section below will add up to ~3,000 real records

N_SYNTHETIC = 7_000
counties = list(COUNTY_PRICE_PARAMS.keys())

# Weight counties by approximate sales volume (larger counties get more records)
county_weights = {
    "Bergen": 0.10, "Monmouth": 0.08, "Essex": 0.08, "Middlesex": 0.08,
    "Hudson": 0.07, "Morris": 0.06, "Union": 0.06, "Ocean": 0.06,
    "Passaic": 0.05, "Somerset": 0.05, "Burlington": 0.04, "Mercer": 0.04,
    "Camden": 0.04, "Atlantic": 0.03, "Cape May": 0.02, "Gloucester": 0.03,
    "Hunterdon": 0.02, "Sussex": 0.02, "Warren": 0.02, "Salem": 0.015, "Cumberland": 0.015,
}
total_weight = sum(county_weights.values())
county_probs = [county_weights[c] / total_weight for c in counties]

# Sample county for each record, then generate
county_samples = rng.choice(counties, size=N_SYNTHETIC, p=county_probs)
synthetic_records = [generate_synthetic_record(county, rng) for county in county_samples]
df_synthetic = pd.DataFrame(synthetic_records)

# Validate all zip codes are 5-character strings
assert (df_synthetic["zip_code"].str.len() == 5).all(), \
    f"Zip code length error: {df_synthetic['zip_code'].str.len().value_counts()}"

print(f"Generated {len(df_synthetic):,} synthetic records.")
print(f"Source breakdown:\n{df_synthetic['source'].value_counts()}")
print(f"Price stats:\n  min=${df_synthetic['price'].min():,.0f}  median=${df_synthetic['price'].median():,.0f}  max=${df_synthetic['price'].max():,.0f}")

In [ ]:
import zipfile
import pathlib

sr1a_path = pathlib.Path(repo_root) / "data" / "raw" / "SR1A_2024.txt"
sr1a_zip_path = pathlib.Path(repo_root) / "data" / "raw" / "SR1A_2024.zip"

df_real = pd.DataFrame()  # default empty

if sr1a_path.exists():
    print(f"Found SR1A file at {sr1a_path}")
    # Attempt to read; adjust sep/engine/widths based on actual file format
    # Try CSV first (comma or pipe separated), fall back to fixed-width
    try:
        df_sr1a_raw = pd.read_csv(sr1a_path, dtype=str, low_memory=False)
    except Exception:
        df_sr1a_raw = pd.read_fwf(sr1a_path, dtype=str)

    print(f"Loaded {len(df_sr1a_raw):,} raw SR1A records. Columns: {list(df_sr1a_raw.columns[:10])}")

    # NOTE: Column names below are PLACEHOLDERS.
    # Inspect df_sr1a_raw.columns after loading and map to actual column names.
    # Consult SR1A_FileLayout_Description.pdf for true field names.
    # The following fields are needed:
    #   - sale_price: numeric sale price
    #   - property_class: class code (class 2 = residential)
    #   - county_code: county identifier
    #   - zip_code: 5-digit zip (or municipality code to derive zip)
    # Adjust the column name strings below to match the actual SR1A column headers.

    SALE_PRICE_COL = "SALE PRICE"        # placeholder -- verify against actual columns
    PROPERTY_CLASS_COL = "PROPERTY CLASS"  # placeholder
    COUNTY_COL = "COUNTY"                  # placeholder
    ZIP_COL = "ZIP CODE"                   # placeholder

    # Filter to residential class 2 and positive sale prices
    available_cols = df_sr1a_raw.columns.tolist()
    print(f"Checking for required columns: {[SALE_PRICE_COL, PROPERTY_CLASS_COL, COUNTY_COL, ZIP_COL]}")

    if all(c in available_cols for c in [SALE_PRICE_COL, PROPERTY_CLASS_COL]):
        df_residential = df_sr1a_raw[
            (df_sr1a_raw[PROPERTY_CLASS_COL].astype(str).str.strip() == "2")
        ].copy()
        df_residential[SALE_PRICE_COL] = pd.to_numeric(df_residential[SALE_PRICE_COL], errors="coerce")
        df_residential = df_residential[df_residential[SALE_PRICE_COL] > 0].copy()

        print(f"Residential records with valid prices: {len(df_residential):,}")

        # Build real records DataFrame
        real_records = []
        county_code_map = {
            "01": "Atlantic", "02": "Bergen", "03": "Burlington", "04": "Camden",
            "05": "Cape May", "06": "Cumberland", "07": "Essex", "08": "Gloucester",
            "09": "Hudson", "10": "Hunterdon", "11": "Mercer", "12": "Middlesex",
            "13": "Monmouth", "14": "Morris", "15": "Ocean", "16": "Passaic",
            "17": "Salem", "18": "Somerset", "19": "Sussex", "20": "Union", "21": "Warren",
        }

        for _, row in df_residential.iterrows():
            price = float(row[SALE_PRICE_COL])
            if price < 100_000 or price > 3_000_000:
                continue  # clip outliers

            county_code = str(row.get(COUNTY_COL, "")).strip().zfill(2)
            county = county_code_map.get(county_code, "Bergen")  # default to Bergen if unknown

            # Zip code -- preserve leading zero
            raw_zip = str(row.get(ZIP_COL, "")).strip()
            zip_code = raw_zip.zfill(5) if raw_zip.isdigit() else "07000"

            # Structural fields: use from MODIV if available (future join), else synthesize
            params = COUNTY_PRICE_PARAMS.get(county, COUNTY_PRICE_PARAMS["Bergen"])
            price_per_sqft = float(rng.uniform(250, 700))
            sqft = int(max(500, min(int(price / price_per_sqft), 8000)))
            bedrooms = min(6, max(1, int(sqft / 500) + int(rng.integers(-1, 2))))
            bathrooms = max(1.0, min(round(float(bedrooms * rng.uniform(0.5, 1.2) * 0.5) * 2) / 2, 5.0))
            lot_size = max(0.05, min(round(float(rng.lognormal(np.log(0.25), 0.8)), 2), 10.0))
            year_built = int(rng.integers(1900, 2025))
            property_type = str(rng.choice(PROPERTY_TYPES, p=PROPERTY_TYPE_PROBS))

            real_records.append({
                "bedrooms": bedrooms,
                "bathrooms": bathrooms,
                "sqft": sqft,
                "lot_size": lot_size,
                "year_built": year_built,
                "zip_code": zip_code,
                "property_type": property_type,
                "price": round(price / 100) * 100,
                "source": "real",
                "county": county,
            })

        df_real = pd.DataFrame(real_records)
        print(f"Processed {len(df_real):,} real records from SR1A.")
    else:
        print(f"WARNING: Required columns not found in SR1A file.")
        print(f"Actual columns: {available_cols}")
        print("Action needed: Update SALE_PRICE_COL, PROPERTY_CLASS_COL, COUNTY_COL, ZIP_COL above to match actual SR1A column names.")
        print("Proceeding with synthetic-only dataset (DATA-04 30% real requirement will NOT be met until SR1A is parsed correctly).")
elif sr1a_zip_path.exists():
    print(f"Found SR1A ZIP at {sr1a_zip_path}. Extract it to data/raw/ and rerun this cell.")
else:
    print("WARNING: SR1A_2024.txt not found in data/raw/.")
    print("To meet DATA-04 (30% real records), download SR1A from:")
    print("  https://www.nj.gov/treasury/taxation/lpt/statdata.shtml")
    print("Place the extracted file at: data/raw/SR1A_2024.txt")
    print("Proceeding with synthetic-only dataset for now.")

print(f"Real records loaded: {len(df_real):,}")

In [ ]:
# Combine synthetic and real records
df_all = pd.concat([df_synthetic, df_real], ignore_index=True) if len(df_real) > 0 else df_synthetic.copy()
df_all = df_all.sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle

# Check DATA-04: at least 30% real records
real_pct = (df_all["source"] == "real").mean() * 100
print(f"Dataset composition: {len(df_all):,} total records")
print(f"  Real:      {(df_all['source'] == 'real').sum():,} ({real_pct:.1f}%)")
print(f"  Synthetic: {(df_all['source'] == 'synthetic').sum():,} ({100-real_pct:.1f}%)")
if real_pct < 30:
    print(f"WARNING: Real records are {real_pct:.1f}% < 30% requirement (DATA-04).")
    print("This is acceptable for development but must be resolved before final training run.")
else:
    print(f"DATA-04 PASSED: {real_pct:.1f}% >= 30% real records.")

# Validate price distribution
validate_price_distribution(df_all, save_path=os.path.join(repo_root, "data", "price_distribution_check.png"))

# Validate zip codes
zip_len_check = df_all["zip_code"].str.len()
assert (zip_len_check == 5).all(), f"Zip code length errors: {zip_len_check.value_counts()}"
print(f"Zip code validation passed: all {len(df_all):,} zip codes are 5 characters.")

# Stratified split by price quartile (70/15/15)
df_all["price_bin"] = pd.qcut(df_all["price"], q=4, labels=False, duplicates="drop")

train_val, test = train_test_split(
    df_all, test_size=0.15, stratify=df_all["price_bin"], random_state=SEED
)
train, val = train_test_split(
    train_val, test_size=0.15/0.85, stratify=train_val["price_bin"], random_state=SEED
)

# Drop helper column
for split in [train, val, test]:
    split.drop(columns=["price_bin"], inplace=True, errors="ignore")

train = train.reset_index(drop=True)
val   = val.reset_index(drop=True)
test  = test.reset_index(drop=True)

total = len(train) + len(val) + len(test)
print(f"\nStratified split complete:")
print(f"  Train: {len(train):,} ({len(train)/total:.1%})")
print(f"  Val:   {len(val):,} ({len(val)/total:.1%})")
print(f"  Test:  {len(test):,} ({len(test)/total:.1%})")
assert abs(len(train)/total - 0.70) < 0.01, f"Train ratio off: {len(train)/total:.3f}"
assert abs(len(val)/total   - 0.15) < 0.01, f"Val ratio off: {len(val)/total:.3f}"
assert abs(len(test)/total  - 0.15) < 0.01, f"Test ratio off: {len(test)/total:.3f}"

In [ ]:
def export_to_jsonl(df: pd.DataFrame, path: str) -> None:
    """Export DataFrame to JSONL with prompt and price fields only."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            prompt = format_prompt(
                bedrooms=int(row["bedrooms"]),
                bathrooms=float(row["bathrooms"]),
                sqft=int(row["sqft"]),
                lot_size=float(row["lot_size"]),
                year_built=int(row["year_built"]),
                zip_code=str(row["zip_code"]),
                property_type=str(row["property_type"]),
            )
            record = {"prompt": prompt, "price": float(row["price"])}
            f.write(json.dumps(record) + "\n")
    print(f"Wrote {len(df):,} records to {path}")

train_path = os.path.join(DATA_SPLITS, "train.jsonl")
val_path   = os.path.join(DATA_SPLITS, "val.jsonl")
test_path  = os.path.join(DATA_SPLITS, "test.jsonl")

export_to_jsonl(train, train_path)
export_to_jsonl(val,   val_path)
export_to_jsonl(test,  test_path)

# Spot-check: read back first record from each split
for path, name in [(train_path, "train"), (val_path, "val"), (test_path, "test")]:
    with open(path) as f:
        first = json.loads(f.readline())
    assert "prompt" in first and "price" in first, f"{name}.jsonl record missing prompt or price"
    assert first["prompt"].endswith("Predicted price: $"), f"{name}.jsonl prompt does not end correctly: {first['prompt'][-40:]!r}"
    assert isinstance(first["price"], float), f"{name}.jsonl price is not float: {type(first['price'])}"
    print(f"{name}.jsonl OK: prompt ends correctly, price is float {first['price']:,.0f}")

print("\nAll splits exported and validated.")
print(f"Data saved to: {DATA_SPLITS}")